In [340]:
import pandas as pd
import numpy as np

In [341]:
# Load and check data
df = pd.read_csv(".../dataset_mood_smartphone_cleaned.csv")
print(df.dtypes)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '.../dataset_mood_smartphone_cleaned.csv'

In [ ]:
# Correct data types
df["date"] = pd.to_datetime(df["date"])
df['time'] = pd.to_datetime(df['time'], format="mixed", errors='coerce')
df['hour'] = df['time'].dt.hour
print(df.dtypes)
df.head()

id                     str
time        datetime64[us]
variable               str
value              float64
date        datetime64[us]
hour                 int32
dtype: object


,id,time,variable,value,date,hour
0,AS14.01,2014-02-26 13:00:00,mood,6.0,2014-02-26,13
1,AS14.01,2014-02-26 15:00:00,mood,6.0,2014-02-26,15
2,AS14.01,2014-02-26 18:00:00,mood,6.0,2014-02-26,18
3,AS14.01,2014-02-26 21:00:00,mood,7.0,2014-02-26,21
4,AS14.01,2014-02-27 09:00:00,mood,6.0,2014-02-27,9


# 0. Pre-aggregation engineering -> Creating two new variables: morning and evening screen time

In [ ]:
# Extract app usage related variables
app_vars = [v for v in df['variable'].unique() if v.startswith('appCat')]

# Divide into morning (6:00 - 10:00) and evening (20:00 - 24:00) times
df_morning = df[(df['variable'].isin(app_vars)) & (df['hour'] >= 6) & (df['hour'] < 10)]
df_evening = df[(df['variable'].isin(app_vars)) & (df['hour'] >= 20) & (df['hour'] < 24)]

# Sum app usage per user per day for morning and evening times
morning_use = df_morning.groupby(['id', 'date'])['value'].sum().reset_index(name='morning_app_use')
evening_use = df_evening.groupby(['id', 'date'])['value'].sum().reset_index(name='evening_app_use')

# 1.1. Aggregating entries into one per individual (id) per day (date)

In [ ]:
# State which variables are to be aggregated by mean vs. sum
mean_vars = ['mood', 'stress', 'activity', 'social', 'circumplex.valence', 'circumplex.arousal']
sum_vars = [v for v in df['variable'].unique() if v not in mean_vars]

df_mean = df[df['variable'].isin(mean_vars)]
df_sum  = df[df['variable'].isin(sum_vars)]

# Aggregate by mean vs. sum separately
df_mean_agg = (
    df_mean.groupby(['id', 'date', 'variable'])['value']
    .mean()
    .reset_index()
)

df_sum_agg = (
    df_sum.groupby(['id', 'date', 'variable'])['value']
    .sum()
    .reset_index()
)

# Concatenate both datasets back together
df_agg = pd.concat([df_mean_agg, df_sum_agg], axis=0)
df_agg.head()

,id,date,variable,value
0,AS14.01,2014-02-26,circumplex.arousal,-0.250000
1,AS14.01,2014-02-26,circumplex.valence,0.750000
2,AS14.01,2014-02-26,mood,6.250000
3,AS14.01,2014-02-27,circumplex.arousal,0.000000
4,AS14.01,2014-02-27,circumplex.valence,0.333333


In [ ]:
# Pivot original data to have one row per id per date, with each variable as its own column
df_day = df_agg.pivot_table(
    index=['id', 'date'],
    columns='variable',
    values='value',
    aggfunc=lambda x: x
).reset_index()
df_day.head()

variable,id,date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,...,appCat.travel,appCat.unknown,appCat.utilities,appCat.weather,call,circumplex.arousal,circumplex.valence,mood,screen,sms
0,AS14.01,2014-02-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN
1,AS14.01,2014-02-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
2,AS14.01,2014-02-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,7.0,NaN,NaN,NaN,NaN,2.0
3,AS14.01,2014-02-20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,3.0
4,AS14.01,2014-02-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0


# 1.2. Mid-aggregation engineering -> Introduce new variable 'emotion' + dd morning/evening app usage + replace NaNs of binary variables with 0s

In [ ]:
# Replace valence and arousal with product as 'emotion'
df_day['emotion'] = df_day['circumplex.valence'] * df_day['circumplex.arousal']
df_day = df_day.drop(columns=['circumplex.valence', 'circumplex.arousal'])

# Add (previously calculated) morning and evening app usage to the daily dataset
df_day = df_day.merge(morning_use, on=['id', 'date'], how='left')
df_day = df_day.merge(evening_use, on=['id', 'date'], how='left')

# Fill missing binary variables with 0 (since the only entries for these have value 1, assume missing means 0)
df_day[['sms', 'call']] = df_day[['sms', 'call']].fillna(0)
df_day.head()

,id,date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,...,appCat.unknown,appCat.utilities,appCat.weather,call,mood,screen,sms,emotion,morning_app_use,evening_app_use
0,AS14.01,2014-02-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,2.0,NaN,NaN,0.0,NaN,NaN,NaN
1,AS14.01,2014-02-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1.0,NaN,NaN,0.0,NaN,NaN,NaN
2,AS14.01,2014-02-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,7.0,NaN,NaN,2.0,NaN,NaN,NaN
3,AS14.01,2014-02-20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,2.0,NaN,NaN,3.0,NaN,NaN,NaN
4,AS14.01,2014-02-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,NaN,NaN,1.0,NaN,NaN,NaN


# 2.1. Define target + aggregate entries into one per individual (id)

In [ ]:
# Identify last day per individual
df_last_days = df_day.groupby("id")["date"].max().reset_index()
df_last_days = df_day.merge(df_last_days, on=["id", "date"])
df_last_days = df_last_days.rename(columns={"date": "last_date"})

# Compute average mood on the last day per individual
df_last_days_mood = (
    df_last_days.groupby("id")
                .agg({
                    "mood": "mean",
                    "last_date": "first"
                })
                .reset_index()
)
df_last_days_mood = df_last_days_mood.rename(columns={"mood": "avg_last_day_mood"})
df_last_days_mood.head()

,id,avg_last_day_mood,last_date
0,AS14.01,NaN,2014-05-05
1,AS14.02,9.000000,2014-04-25
2,AS14.03,NaN,2014-05-08
3,AS14.05,6.333333,2014-05-05
4,AS14.06,7.000000,2014-05-08


In [ ]:
# Merge last day mood back to daily dataset and filter for prior days only
df_prior_days = df_day.merge(df_last_days_mood[["id", "last_date"]], on="id")
df_prior_days = df_prior_days[df_prior_days['date'] < df_prior_days['last_date']]

# Compute time decay weights
decay_rate = 0.1
df_prior_days["diff_days"] = (df_prior_days["last_date"] - df_prior_days["date"]).dt.days
df_prior_days["weight"] = np.exp(-decay_rate * df_prior_days["diff_days"])

value_cols = [c for c in df_prior_days.columns if c not in ["id", "date", "last_date", "diff_days", "weight"]]
for col in value_cols:
    df_prior_days[col] = df_prior_days[col] * df_prior_days["weight"]

df_prior_days.head()

,id,date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,appCat.other,...,call,mood,screen,sms,emotion,morning_app_use,evening_app_use,last_date,diff_days,weight
0,AS14.01,2014-02-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000906,NaN,NaN,0.000000,NaN,NaN,NaN,2014-05-05,77,0.000453
1,AS14.01,2014-02-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000500,NaN,NaN,0.000000,NaN,NaN,NaN,2014-05-05,76,0.000500
2,AS14.01,2014-02-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.003872,NaN,NaN,0.001106,NaN,NaN,NaN,2014-05-05,75,0.000553
3,AS14.01,2014-02-20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.001223,NaN,NaN,0.001834,NaN,NaN,NaN,2014-05-05,74,0.000611
4,AS14.01,2014-02-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,NaN,NaN,0.000676,NaN,NaN,NaN,2014-05-05,73,0.000676


In [ ]:
df_weighted_avg = df_prior_days.groupby("id").agg(
    **{
        col: (col, lambda x: (
            x.sum() / df_prior_days.loc[x.index[x.notna()], "weight"].sum()
            if x.notna().any() else np.nan
        ))
        for col in value_cols
    }
).reset_index()

df_non_ts = df_last_days_mood.merge(df_weighted_avg, on="id", how="left")
df_non_ts = df_non_ts.dropna(subset=['avg_last_day_mood'])
df_non_ts.head()

,id,avg_last_day_mood,last_date,activity,appCat.builtin,appCat.communication,appCat.entertainment,appCat.finance,appCat.game,appCat.office,...,appCat.unknown,appCat.utilities,appCat.weather,call,mood,screen,sms,emotion,morning_app_use,evening_app_use
1,AS14.02,9.000000,2014-04-25,0.239149,862.627855,1356.925203,388.461668,NaN,NaN,NaN,...,NaN,29.168357,NaN,7.079777,6.595828,2280.873833,1.314698,0.143823,524.779100,752.140588
3,AS14.05,6.333333,2014-05-05,0.046840,527.572259,3193.958698,317.561224,NaN,NaN,21.127000,...,107.118652,34.862635,NaN,0.890382,7.027135,4271.678528,1.161513,-0.267457,268.765287,1098.301208
4,AS14.06,7.000000,2014-05-08,0.152735,466.906657,2841.931963,1237.333201,NaN,454.629781,91.185955,...,93.997000,NaN,11.174978,1.846620,7.083244,5605.556281,0.488498,-0.208280,1043.028210,1412.055369
5,AS14.07,5.500000,2014-05-05,0.062493,1008.998370,2761.525609,835.815287,NaN,14.065000,NaN,...,132.975930,20.683939,NaN,1.650400,5.791656,3983.158571,1.332691,-0.137749,757.867740,2177.653508
6,AS14.08,6.666667,2014-05-05,0.045981,104.517577,834.765413,172.255122,NaN,NaN,84.703914,...,133.402545,18.950239,NaN,0.127060,6.617148,1373.739571,0.267306,-0.102887,270.050235,499.429164


In [ ]:
df_weighted_avg.to_csv(".../dataset_mood_smartphone_engineered_ts.csv", index=False)
df_non_ts.to_csv(".../dataset_mood_smartphone_engineered_non_ts.csv", index=False)